# 07 — Qwen2.5-VL-3B + LoRA sur le modèle de langue **et l'encodeur visuel**

**Une seule différence avec le notebook 05 (`ECHELLE = 1`, CER 22,1 %)** : les adaptateurs LoRA sont aussi ajoutés à l'encodeur visuel (blocs ViT et projecteur qui relie la vision au modèle de langue). Images à leur taille d'origine, mêmes réglages d'entraînement (lots de 2 × 8 pas d'accumulation, 2 époques, taux d'apprentissage 2e-4 cosinus, LoRA r = 16, α = 32).

**Hypothèse** : les analyses d'erreurs (notebook 06) montrent que Qwen perd surtout sur les lettres ordinaires et que l'agrandissement des images aggrave le problème. Si la cause est un encodeur visuel inadapté aux écritures médiévales, l'adapter doit faire baisser **les substitutions** et en particulier **les confusions vers `i`** (254 avec le notebook 05).

**Environnement** : Kaggle, Accelerator *GPU T4 x2*, Internet activé, dataset `lignes-occitan-poly` en Input.
1. Lancer les cellules 1 à 5 en session interactive : la cellule 5 vérifie la mémoire.
2. Si la mémoire est sous 14 Go : **Save Version → Save & Run All (Commit) → Save**, puis vérifier qu'une nouvelle version apparaît avec le statut *Running*, et arrêter la session interactive.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                            # un seul GPU : pas de DataParallel
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # limite la fragmentation mémoire

!pip uninstall -y -q torchao          # version préinstallée incompatible avec peft
!pip install -q peft jiwer

# "Batch" pendant un Save & Run All, "Interactive" dans l'éditeur (vérifié lors de l'expérience ×2)
INTERACTIF = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive") == "Interactive"
print("Session interactive :", INTERACTIF)

In [ ]:
import json, csv, time, unicodedata, torch
from pathlib import Path
from collections import defaultdict
from PIL import Image

print("GPU :", torch.cuda.get_device_name(0), "| visibles :", torch.cuda.device_count())

DATA = next(Path("/kaggle/input").rglob("lignes.jsonl")).parent
tout = [json.loads(l) for l in open(DATA / "lignes.jsonl", encoding="utf-8")]
splits = {s: [e for e in tout if e["split"] == s] for s in ("train", "dev", "test")}
print({s: len(v) for s, v in splits.items()})   # train 3392, dev 327, test 574

# Sur Kaggle, les noms de fichiers accentués sont stockés en NFD : on les indexe en NFC
INDEX = {unicodedata.normalize("NFC", p.name): p for p in (DATA / "images").iterdir()}
def chemin_image(e):
    return INDEX[unicodedata.normalize("NFC", Path(e["image"]).name)]
assert all(unicodedata.normalize("NFC", Path(e["image"]).name) in INDEX for e in tout)

## Modèle + LoRA (modèle de langue + encodeur visuel)

Couches visées :
- **modèle de langue** : `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` (comme dans le notebook 05) ;
- **encodeur visuel** : dans chaque bloc, l'attention (`attn.qkv`, `attn.proj`) et le MLP (`mlp.gate_proj`, `mlp.up_proj`, `mlp.down_proj`) ;
- **projecteur** (`merger`, ses deux couches linéaires `mlp.0` et `mlp.2`) : le petit réseau qui traduit les tokens d'image en entrées pour le modèle de langue.

La cellule vérifie que des adaptateurs ont bien été posés dans les trois parties, et s'arrête sinon.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODELE = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(MODELE)
model = AutoModelForImageTextToText.from_pretrained(MODELE, dtype=torch.float16, device_map="cuda:0")
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

CIBLES = (
    r".*("
    r"language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
    r"|visual\.blocks\.\d+\.(attn\.qkv|attn\.proj|mlp\.gate_proj|mlp\.up_proj|mlp\.down_proj)"
    r"|visual\.merger\.mlp\.(0|2)"
    r")"
)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                         task_type="CAUSAL_LM", target_modules=CIBLES))
model.print_trainable_parameters()

noms = [n for n, _ in model.named_modules() if n.endswith("lora_A")]
compte = {
    "modèle de langue": sum("language_model" in n for n in noms),
    "encodeur visuel":  sum("visual.blocks" in n for n in noms),
    "projecteur":       sum("visual.merger" in n for n in noms),
}
print(compte)
if compte["encodeur visuel"] == 0 or compte["projecteur"] == 0:
    print([n for n, _ in model.named_modules() if "visual" in n][:40])
    raise SystemExit("Aucun adaptateur dans la partie visuelle : noms de couches différents, voir la liste ci-dessus.")

## Préparation des exemples (identique au notebook 05)

In [ ]:
CONSIGNE = (
    "Transcris exactement cette ligne d'un manuscrit médiéval en occitan. "
    "Garde les abréviations telles qu'elles sont écrites, sans les développer. "
    "Réponds uniquement par la transcription."
)

def preparer(e):
    image = Image.open(chemin_image(e)).convert("RGB")
    question = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": CONSIGNE},
    ]}]
    complet = question + [{"role": "assistant", "content": [{"type": "text", "text": e["text"]}]}]
    entree = processor.apply_chat_template(complet, tokenize=True, return_dict=True, return_tensors="pt")
    debut = processor.apply_chat_template(question, add_generation_prompt=True, tokenize=True,
                                          return_dict=True, return_tensors="pt")
    labels = entree["input_ids"].clone()
    labels[:, :debut["input_ids"].shape[1]] = -100   # on n'apprend que la réponse
    entree["labels"] = labels
    return entree

def assembler(lot):
    exemples = [preparer(e) for e in lot]
    longueur = max(x["input_ids"].shape[1] for x in exemples)
    pad = processor.tokenizer.pad_token_id
    def completer(t, valeur):
        return torch.nn.functional.pad(t, (0, longueur - t.shape[1]), value=valeur)
    return {
        "input_ids":      torch.cat([completer(x["input_ids"], pad) for x in exemples]),
        "attention_mask": torch.cat([completer(x["attention_mask"], 0) for x in exemples]),
        "labels":         torch.cat([completer(x["labels"], -100) for x in exemples]),
        "pixel_values":   torch.cat([x["pixel_values"] for x in exemples]),
        "image_grid_thw": torch.cat([x["image_grid_thw"] for x in exemples]),
    }

ids = assembler(splits["train"][:1])["labels"][0]
print(repr(processor.tokenizer.decode(ids[ids != -100])), "| vrai :", repr(splits["train"][0]["text"]))

## Test de mémoire (session interactive uniquement)

Les gradients traversent maintenant aussi l'encodeur visuel : la mémoire monte. Un pas d'entraînement complet, avec l'optimiseur, sur les 16 plus grandes lignes du train, par lots de 2 comme pour le vrai entraînement. Ignoré pendant un Save & Run All.

In [ ]:
from transformers import TrainingArguments, Trainer

LOT, ACCUMULATION = 2, 8      # comme le notebook 05 ; passer à 1, 16 si la mémoire dépasse 14 Go

if INTERACTIF:
    surface = lambda e: (lambda im: im.width * im.height)(Image.open(chemin_image(e)))
    plus_grandes = sorted(splits["train"], key=lambda e: -surface(e))[:16]
    torch.cuda.reset_peak_memory_stats()
    model.config.use_cache = False
    Trainer(
        model=model,
        args=TrainingArguments(output_dir="/kaggle/working/test_memoire", per_device_train_batch_size=LOT,
                               gradient_accumulation_steps=ACCUMULATION, max_steps=1, fp16=True,
                               save_strategy="no", report_to="none", remove_unused_columns=False),
        train_dataset=plus_grandes, data_collator=assembler,
    ).train()
    print(f"Mémoire max : {torch.cuda.max_memory_allocated() / 1e9:.1f} Go (limite ≈ 14,5)")
    print("Sous 14 Go : Save Version → Save & Run All. Ne lance pas les cellules suivantes ici.")
else:
    print("Exécution en arrière-plan : test ignoré.")

## Entraînement (à lancer via Save & Run All)

In [ ]:
if INTERACTIF:
    raise SystemExit("Session interactive : lance l'entraînement avec Save Version → Save & Run All.")

model.config.use_cache = False
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/kaggle/working/qwen_occitan_vision",
        per_device_train_batch_size=LOT, per_device_eval_batch_size=LOT,
        gradient_accumulation_steps=ACCUMULATION,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=20,
        num_train_epochs=2, fp16=True, logging_steps=10,
        eval_strategy="steps", eval_steps=100,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
    ),
    train_dataset=splits["train"], eval_dataset=splits["dev"], data_collator=assembler,
)
t0 = time.time()
trainer.train()
print(f"Durée d'entraînement : {(time.time() - t0) / 3600:.1f} h")
trainer.save_model("/kaggle/working/qwen_occitan_lora_vision")

## Évaluation sur le test

In [ ]:
import jiwer
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein   # installé avec jiwer

model.eval()
model.config.use_cache = True

def transcrire(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": CONSIGNE},
    ]}]
    entrees = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                            return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        sortie = model.generate(**entrees, max_new_tokens=100, do_sample=False)
    return processor.decode(sortie[0, entrees["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def nfd(s):
    return unicodedata.normalize("NFD", s or "").strip()

lignes = [{"image": e["image"], "manuscrit": e["manuscrit"], "vrai": e["text"],
           "pred": transcrire(Image.open(chemin_image(e)).convert("RGB"))}
          for e in tqdm(splits["test"])]

with open("/kaggle/working/predictions_qwen_lora_vision.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["image", "manuscrit", "vrai", "pred"])
    w.writeheader(); w.writerows(lignes)

vrais = [nfd(r["vrai"]) for r in lignes]
preds = [nfd(r["pred"]) for r in lignes]
print(f"CER global : {jiwer.cer(vrais, preds):.1%}   (notebook 05 : 22,1 %)")
par_ms = defaultdict(lambda: ([], []))
for r, v, p in zip(lignes, vrais, preds):
    par_ms[r["manuscrit"]][0].append(v); par_ms[r["manuscrit"]][1].append(p)
for m, (v, p) in par_ms.items():
    print(f"  {m:25} {len(v):4} lignes   CER {jiwer.cer(v, p):.1%}")

# Vérification de l'hypothèse : substitutions et confusions vers « i »
n = sum(len(v) for v in vrais)
subs, vers_i = 0, 0
for v, p in zip(vrais, preds):
    for op in Levenshtein.editops(v, p):
        if op.tag == "replace":
            subs += 1
            vers_i += p[op.dest_pos] == "i" and v[op.src_pos] != "i"
print(f"\nSubstitutions : {subs / n:.1%}   (notebook 05 : 13,5 %)")
print(f"Confusions vers « i » : {vers_i}   (notebook 05 : 254)")